In [ ]:
pip install pandas matplotlib geopandas
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("data")
OUT = Path("figures")
OUT.mkdir(exist_ok=True)

AGE_ORDER = ["0-14", "15-24", "25-34", "35-44", "45-54", "55-64", "65 and above"]

def save_show(fig, filename):
    fig.tight_layout()
    fig.savefig(OUT / filename, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

# FIGURE 1: Maps of Africa/Nigeria and Abuja/Ibadan
def plot_figure1_maps(
    africa_geojson=None,
    nigeria_geojson=None,
    nigeria_name_field="NAME",
    state_geojson=None,
    state_name_field="state"
):

    import geopandas as gpd

    if not all([africa_geojson, nigeria_geojson, state_geojson]):
        raise ValueError(
            "Provide paths to Africa, Nigeria and Nigerian administrative "
            "boundary GeoJSON/shapefiles. The paper does not supply these files."
        )

    africa = gpd.read_file(africa_geojson)
    nigeria = gpd.read_file(nigeria_geojson)
    admin = gpd.read_file(state_geojson)

    fig, axes = plt.subplots(1, 2, figsize=(13, 6))

    # Panel A: Africa with Nigeria highlighted
    africa.plot(ax=axes[0], color="lightgrey", edgecolor="white", linewidth=0.4)
    nigeria.to_crs(africa.crs).plot(
        ax=axes[0], color="green", edgecolor="black", linewidth=0.7
    )
    axes[0].set_title("(a) Location of Nigeria in Africa")
    axes[0].axis("off")

    # Panel B: Nigeria with Abuja/FCT and Oyo State highlighted
    nigeria.plot(ax=axes[1], color="whitesmoke", edgecolor="black")
    admin.plot(ax=axes[1], color="lightgrey", edgecolor="white", linewidth=0.3)

    # Edit these names to match the exact labels in the chosen boundary file.
    if state_name_field in admin.columns:
        selected = admin[
            admin[state_name_field].astype(str).str.contains(
                r"Federal Capital|FCT|Oyo", case=False, regex=True
            )
        ]
        if not selected.empty:
            selected.plot(ax=axes[1], color="magenta", edgecolor="black")

    axes[1].set_title("(b) Abuja (FCT) and Ibadan (Oyo State)")
    axes[1].axis("off")

    save_show(fig, "figure1_maps.png")


# FIGURE 2: Comparative population pyramids

def population_pyramid(ax, df, male_col, female_col, title):
    d = df.copy()
    d["age_group"] = pd.Categorical(
        d["age_group"], categories=AGE_ORDER, ordered=True
    )
    d = d.sort_values("age_group")

    y = range(len(d))
    ax.barh(y, -d[male_col], label="Male")
    ax.barh(y, d[female_col], label="Female")
    ax.set_yticks(list(y))
    ax.set_yticklabels(d["age_group"].astype(str))
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel("Population")
    ax.set_ylabel("Age group")
    ax.legend()

def plot_figure2_population_pyramids():
    df = pd.read_csv(DATA / "population.csv")
    fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharey=True)

    population_pyramid(
        axes[0], df, "ibadan_male", "ibadan_female", "Ibadan"
    )
    population_pyramid(
        axes[1], df, "abuja_male", "abuja_female", "Abuja"
    )

    fig.suptitle("Comparative Population Pyramids of Ibadan and Abuja")
    save_show(fig, "figure2_population_pyramids.png")

# FIGURE 3: Basis of diagnosis pie charts

def plot_figure3_diagnosis_pies():
    df = pd.read_csv(DATA / "diagnosis.csv")
    registries = ["Ibadan", "Abuja"]

    fig, axes = plt.subplots(2, 1, figsize=(8, 11))

    for ax, registry in zip(axes, registries):
        subset = df[df["registry"].str.lower() == registry.lower()]
        if subset.empty:
            raise ValueError(f"No diagnosis data found for {registry}.")

        ax.pie(
            subset["count"],
            labels=subset["basis"],
            autopct="%1.1f%%",
            startangle=90
        )
        ax.set_title(f"Basis of diagnosis in {registry} Cancer Registry")

    save_show(fig, "figure3_basis_of_diagnosis.png")

# FIGURE 4: Age-specific rates for all cancers, men and women
def plot_figure4_age_specific_all_cancers():
    df = pd.read_csv(DATA / "age_specific_all.csv")
    sexes = ["Men", "Women"]

    fig, axes = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

    for ax, sex in zip(axes, sexes):
        subset = df[df["sex"].str.lower() == sex.lower()].copy()
        if subset.empty:
            raise ValueError(f"No age-specific data found for {sex}.")

        subset["age_group"] = pd.Categorical(
            subset["age_group"], categories=AGE_ORDER, ordered=True
        )
        subset = subset.sort_values("age_group")

        for series, line_data in subset.groupby("series"):
            ax.plot(
                line_data["age_group"].astype(str),
                line_data["rate"],
                marker="o",
                linewidth=1.8,
                label=series
            )

        ax.set_title(f"Age-specific rates for all cancers, {sex.lower()}")
        ax.set_ylabel("Age-specific rate per 100,000")
        ax.grid(True, alpha=0.3)
        ax.legend()

    axes[-1].set_xlabel("Age group")
    save_show(fig, "figure4_age_specific_all_cancers.png")

# FIGURE 5: Age-specific incidence rates for breast, cervical, prostate

def plot_figure5_breast_cervical_prostate():
    df = pd.read_csv(DATA / "age_specific_cancers.csv")
    cancers = ["Breast", "Cervical", "Prostate"]

    fig, axes = plt.subplots(3, 1, figsize=(10, 14), sharex=True)

    for ax, cancer in zip(axes, cancers):
        subset = df[df["cancer"].str.lower() == cancer.lower()].copy()
        if subset.empty:
            raise ValueError(f"No age-specific data found for {cancer} cancer.")

        subset["age_group"] = pd.Categorical(
            subset["age_group"], categories=AGE_ORDER, ordered=True
        )
        subset = subset.sort_values("age_group")

        # Lines named "Ibadan" and "Abuja" are point estimates.
        for series, line_data in subset.groupby("series"):
            ax.plot(
                line_data["age_group"].astype(str),
                line_data["rate"],
                marker="o",
                linewidth=1.8,
                label=series
            )

        ax.set_title(f"Age-specific rates for {cancer.lower()} cancer")
        ax.set_ylabel("Age-specific rate per 100,000")
        ax.grid(True, alpha=0.3)
        ax.legend()

    axes[-1].set_xlabel("Age group")
    save_show(fig, "figure5_site_specific_cancers.png")



SyntaxError: invalid syntax (595959739.py, line 1)